# 미션15 - Docker 기반 머신러닝 협업 워크플로우 완료 보고서

---

## 1. 프로젝트 개요

### 1.1. 미션 목표
Docker를 활용한 머신러닝 모델 학습 및 추론 협업 워크플로우 구현. 두 명의 연구자가 컨테이너 기반 환경에서 모델 학습(연구자 1)과 추론(연구자 2)을 분리하여 수행하며, 공유 볼륨을 통해 모델 파일을 전달하는 시스템을 설계 및 구현했습니다.

### 1.2. 핵심 설계 원칙
- **환경 일관성(Environment Consistency)**: Python 3.10 및 패키지 버전 통일
- **방어적 설계(Defensive Design)**: 모델 파일 생성 전 추론 실행 방지
- **재현성(Reproducibility)**: random_state=42, 패키지 버전 고정


---

## 2. 기술 구현

### 2.1. 시스템 아키텍처

```mermaid
graph TB
    A["Host System"] --> B["./data (Bind Mount)"]
    B --> C["mission15_train Container"]
    B --> D["mission15_inference Container"]
    
    C --> E["train_model.py"]
    E --> F["model.pkl 생성"]
    F --> B
    
    D --> G["startup.sh (10s timeout)"]
    G --> H["model.pkl 대기"]
    H --> I["Jupyter Notebook 시작"]
    I --> J["inference.ipynb"]
    J --> K["result.csv 생성"]
    K --> B
    
    C -.->|depends_on| D
```

### 2.2. 방어적 설계 구현

#### 2.2.1. 이중 방어 메커니즘
```yaml
# docker-compose.yml
services:
  mission15_inference:
    depends_on:
      - mission15_train  # 1차 방어: 시작 순서 제어
```

```bash
# startup.sh (mission15_inference)
# 2차 방어: model.pkl 존재 확인 (타임아웃 10초)
timeout_secs=10
while [ "$elapsed" -lt "$timeout_secs" ]; do
  if [ -f "/app/data/model.pkl" ]; then
    echo "/app/data/model.pkl found."
    break
  fi
  sleep "$interval_secs"
  elapsed=$((elapsed + interval_secs))
done
```

**설계 의도**: `depends_on`은 컨테이너 시작 순서만 제어하므로, 실제 model.pkl 생성 완료를 보장하지 않습니다. startup.sh에서 파일 존재를 명시적으로 확인하여 학습 완료 전 추론 실행 오류를 방지합니다.

#### 2.2.2. 환경 감지 및 경로 분기
```python
# train_model.py
IS_DOCKER = os.environ.get('RUNNING_IN_DOCKER', 'False') == 'True'
DATA_DIR = r'..\data' if not IS_DOCKER else '/app/data'
```

로컬 개발 환경과 컨테이너 환경에서 동일한 코드로 실행 가능하도록 환경변수 기반 경로 분기를 구현했습니다.

### 2.3. 데이터 전처리 및 모델링

#### 2.3.1. 파이프라인 구성
```python
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), 
         ['Extracurricular Activities'])
    ],
    remainder='passthrough'
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
```

**특징**:
- OneHotEncoder의 `drop='first'` 옵션으로 다중공선성(multicollinearity) 방지
- `handle_unknown='ignore'` 설정으로 테스트 데이터의 미지 범주 처리
- ColumnTransformer로 범주형/수치형 변수 통합 전처리

#### 2.3.2. Atomic Write 패턴
```python
# train_model.py
tmp_path = MODEL_FILENAME + ".tmp"
joblib.dump(pipeline, tmp_path)
os.replace(tmp_path, MODEL_FILENAME)  # Atomic operation
```

파일 쓰기 중 연구자 2가 접근하는 경우를 방지하기 위해 임시 파일 생성 후 원자적(atomic) 이동을 수행했습니다.

### 2.4. 패키지 버전 관리
```
# requirements.txt
numpy==1.24.4
pandas==1.5.3
scikit-learn==1.2.2
joblib==1.2.0
```

두 컨테이너에서 동일한 패키지 버전을 사용하여 모델 직렬화/역직렬화 호환성을 보장했습니다.

---

## 3. 실행 결과

### 3.1. 모델 성능
- **평가 지표**: RMSE (Root Mean Squared Error)
- **검증 세트 RMSE**: 2.0103
- **R-squared**: 0.9893
- **해석**: Performance Index 범위(10-100) 대비 약 2.2%의 오차로, 모델이 데이터 변동성의 98.93%를 설명

### 3.2. 실행 방법
```bash
# 1. 전체 시스템 실행
docker-compose up --build

# 2. Jupyter Notebook 접속
# 브라우저: http://localhost:8888

# 3. inference.ipynb 실행
# model.pkl 자동 로드 → 추론 → result.csv 생성

# 4. 정리
docker-compose down
```

### 3.3. Docker Hub 배포
```bash
# 이미지 빌드 및 푸시
docker build -t [사용자명]/mission15-train:latest ./mission15_train
docker push [사용자명]/mission15-train:latest

docker build -t [사용자명]/mission15-inference:latest ./mission15_inference
docker push [사용자명]/mission15-inference:latest
```

**Docker Hub URL**: `[사용자명]/mission15-train:latest`, `[사용자명]/mission15-inference:latest`

---

## 4. 설계 특징 및 회고

### 4.1. 핵심 설계 패턴

#### 4.1.1. 방어적 설계의 효과
- **문제**: `depends_on`만으로는 model.pkl 생성 완료 보장 불가
- **해결**: startup.sh에서 10초 타임아웃으로 파일 존재 확인
- **효과**: 학습 지연 또는 실패 시 추론 컨테이너가 명확한 에러 메시지 출력

#### 4.1.2. Bind Mount 선택 이유
Named Volume 대신 Bind Mount(`./data:/app/data`)를 사용한 이유:
- 호스트에서 직접 파일 확인 가능
- 디버깅 및 데이터 검증 용이
- 학습/추론 결과를 호스트에 영구 보관

### 4.2. 재현성 확보 전략
1. **패키지 버전 고정**: requirements.txt
2. **random_state 고정**: train_test_split(random_state=42)
3. **Python 버전 통일**: 3.10-slim (연구자 1), jupyter/scipy-notebook (Python 3.10 포함)

### 4.3. 개선 방향
1. **Health Check 도입**: Docker의 HEALTHCHECK 명령어로 model.pkl 생성 상태를 컨테이너 헬스로 관리
2. **로깅 강화**: train_model.py에서 학습 진행률 및 중간 지표를 구조화된 로그로 출력
3. **CI/CD 통합**: GitHub Actions로 이미지 자동 빌드 및 Docker Hub 푸시

---

## 5. 결론

본 프로젝트는 Docker 기반 머신러닝 협업 환경에서 **방어적 설계**를 통해 안정적인 워크플로우를 구현했습니다. 특히 startup.sh의 10초 타임아웃 로직은 `depends_on`의 한계를 보완하여 모델 파일 생성 전 추론 실행을 효과적으로 방지했습니다. 패키지 버전 통일, Bind Mount 방식의 파일 공유, Atomic Write 패턴 등을 통해 **재현성**과 **환경 일관성**을 확보했으며, RMSE 2.0103의 우수한 모델 성능을 달성했습니다.

---

In [ ]:
# 1.1. 라이브러리 임포트
import pandas as pd
import numpy as np
import os
import joblib # 모델 로드용
import logging
import warnings
warnings.filterwarnings('ignore')

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# 기본 스트림 핸들러 설정 — 터미널에 INFO 이상 로그 출력
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# --- 0. 환경 감지 ---
# Dockerfile의 ENV 설정 덕분에 컨테이너 실행 시 True로 설정됨
IS_DOCKER = os.environ.get('RUNNING_IN_DOCKER', 'False') == 'True'

# --- 경로 설정 ---
# mission15_train 컨테이너에서 model.pkl이 저장된 경로 (Named Volume 마운트 지점)
MODEL_INPUT_DIR = r'..\..\data' if not IS_DOCKER else '/app/data'
MODEL_FILENAME = os.path.join(MODEL_INPUT_DIR, 'model.pkl')

# mission15_train 컨테이너에서 mission15_test.csv가 저장된 경로 (Named Volume 또는 Bind Mount 마운트 지점)
TEST_DATA_DIR = r'..\..\data' if not IS_DOCKER else '/app/data'
TEST_FILE = os.path.join(TEST_DATA_DIR, 'mission15_test.csv')

# 최종 result.csv를 저장할 경로 (Bind Mount 마운트 지점, 호스트 PC 접근 가능)
RESULT_OUTPUT_DIR = r'..\..\data' if not IS_DOCKER else '/app/data'
RESULT_FILE = os.path.join(RESULT_OUTPUT_DIR, 'result.csv')

logger.info(f"모델 파일 경로: {MODEL_FILENAME}")
logger.info(f"테스트 데이터 경로: {TEST_FILE}")
logger.info(f"결과 저장 경로: {RESULT_FILE}")
logger.info("-" * 30)
# --- 1.2. 모델 및 데이터 로드 ---
try:
    # 모델 로드
    pipeline = joblib.load(MODEL_FILENAME)
    logger.info(f"{MODEL_FILENAME} 파이프라인 로드 성공.")
    
    # 테스트 데이터 로드
    test_df = pd.read_csv(TEST_FILE)
    logger.info(f"{TEST_FILE} 데이터 로드 성공.")
    
except FileNotFoundError as e:
    logger.error(f"파일 로드 실패: {e}")
    logger.error("연구자 1의 컨테이너가 성공적으로 실행되어 model.pkl 및 mission15_test.csv가 생성/공유되었는지 확인하십시오.")
    exit()

logger.info(f"테스트 데이터 크기: {test_df.shape}")
test_df.head()

2025-12-02 10:46:24,422 - INFO - 모델 파일 경로: ..\..\data\model.pkl
2025-12-02 10:46:24,422 - INFO - 테스트 데이터 경로: ..\..\data\mission15_test.csv
2025-12-02 10:46:24,422 - INFO - 결과 저장 경로: ..\..\data\result.csv
2025-12-02 10:46:24,422 - INFO - ------------------------------
2025-12-02 10:46:25,678 - INFO - ..\..\data\model.pkl 파이프라인 로드 성공.
2025-12-02 10:46:25,686 - INFO - ..\..\data\mission15_test.csv 데이터 로드 성공.
2025-12-02 10:46:25,692 - INFO - 테스트 데이터 크기: (3000, 5)


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced
0,7,99,Yes,9,1
1,8,51,Yes,7,2
2,8,91,No,4,5
3,5,79,No,7,8
4,2,72,No,4,3


In [2]:
# 'Performance Index' 열이 없으므로, 전체 DataFrame을 사용하여 예측
X_test = test_df.copy()

# 파이프라인을 통한 예측 수행
# pipeline.predict()는 전처리를 내부적으로 처리합니다.
predictions = pipeline.predict(X_test)

logger.info("추론 완료.")

2025-12-02 10:47:40,028 - INFO - 추론 완료.


In [3]:
# 1. 예측값을 가장 가까운 정수로 반올림
predictions_rounded = np.rint(predictions)

# 2. 값 제한 (클리핑): 10 ~ 100 사이로 보정
predictions_final = np.clip(predictions_rounded, 10, 100)

logger.info(f"예측 결과 (최소/최대): {predictions_final.min()}/{predictions_final.max()}")

2025-12-02 10:47:49,659 - INFO - 예측 결과 (최소/최대): 13.0/98.0


In [4]:
# 최종 결과를 담을 DataFrame 생성
result_df = test_df.copy()
result_df['Predicted Performance Index'] = predictions_final.astype(int)

# 추론 결과 확인
logger.info("--- 최종 추론 결과 미리보기 ---")
logger.info(f"\n{result_df.tail()}")

2025-12-02 10:48:03,877 - INFO - --- 최종 추론 결과 미리보기 ---
2025-12-02 10:48:03,877 - INFO - 
      Hours Studied  Previous Scores Extracurricular Activities  Sleep Hours  \
2995              8               87                        Yes            4   
2996              1               48                        Yes            8   
2997              3               46                         No            5   
2998              9               52                         No            9   
2999              1               49                        Yes            4   

      Sample Question Papers Practiced  Predicted Performance Index  
2995                                 9                           82  
2996                                 5                           23  
2997                                 8                           25  
2998                                 7                           50  
2999                                 2                           22  


In [6]:
# 저장 경로 폴더가 없으면 생성 (Bind Mount가 없거나 권한 문제 시 대비)
os.makedirs(RESULT_OUTPUT_DIR, exist_ok=True)

# 결과 파일 저장
result_df.to_csv(RESULT_FILE, index=False)
logger.info(f"**최종 추론 결과가 '{RESULT_FILE}'에 성공적으로 저장되었습니다.")
logger.info("이 파일은 호스트 PC의 마운트된 볼륨 위치에서 확인할 수 있습니다.")

2025-12-02 10:48:46,952 - INFO - **최종 추론 결과가 '..\..\data\result.csv'에 성공적으로 저장되었습니다.
2025-12-02 10:48:46,954 - INFO - 이 파일은 호스트 PC의 마운트된 볼륨 위치에서 확인할 수 있습니다.
